# Constellation Software (CSU) — Deal Tracker

**Purpose:** Scrape and analyze Constellation Software's acquisition press releases to track M&A cadence, operating group activity, and deal velocity as a real-time signal for capital deployment and organic growth blend.

**Data source:** `csisoftware.com/investor-relations/press-releases`  
**Coverage:** All acquisition announcements by operating group (Harris, Volaris, Jonas, Vela, Perseus, Topicus)

---
### Signal framework
| Signal | Threshold | Investment implication |
|--------|-----------|------------------------|
| Accelerating | >25 deals/quarter (TTM) | Capital deployment expanding; watch for ROIC dilution |
| On-pace | 15–25 deals/quarter (TTM) | Consistent with historical 60–80 deals/year |
| Slowing | <15 deals/quarter (TTM) | Pipeline tightening or valuation discipline increasing |
| Vertical watch | Single vertical >30% of deals | Concentration risk or category dominance forming |

*CSI has completed ~100+ acquisitions per year in recent years. Pace slowdown often precedes FCF step-up as integration costs normalize.*

## 1. Setup & Dependencies

In [ ]:
# Install required packages (uncomment if needed)
# !pip install requests beautifulsoup4 pandas matplotlib lxml

import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import re
from datetime import datetime, date
from collections import defaultdict
import time
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 80)
print('Libraries loaded.')

## 2. Press Release Scraper

Fetch all acquisition announcements from CSI's investor relations press release page.
Each press release title is parsed for operating group, target company name, and announcement date.

In [ ]:
# --- Configuration ---
BASE_URL = 'https://www.csisoftware.com'
PR_URL   = 'https://www.csisoftware.com/investor-relations/press-releases'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (research/investing; academic use)',
    'Accept-Language': 'en-US,en;q=0.9',
}

# Known CSI operating groups and common aliases in press release titles
OPERATING_GROUPS = {
    'Harris':   ['Harris'],
    'Volaris':  ['Volaris'],
    'Jonas':    ['Jonas'],
    'Vela':     ['Vela'],
    'Perseus':  ['Perseus'],
    'Topicus':  ['Topicus', 'TOI'],
    'Lumine':   ['Lumine'],
    'Modaxo':   ['Modaxo'],
}

# Regex: match acquisition-related titles
ACQ_PATTERN = re.compile(
    r'acqui|purchase|acquire|completion.*acqui|closed.*acqui',
    re.IGNORECASE
)

In [ ]:
def get_press_release_pages(session, base_pr_url, max_pages=30):
    """Paginate through CSI press releases and return list of (date, title, url) tuples."""
    results = []
    page = 1

    while page <= max_pages:
        url = base_pr_url if page == 1 else f'{base_pr_url}?page={page}'
        try:
            resp = session.get(url, headers=HEADERS, timeout=15)
            resp.raise_for_status()
        except Exception as e:
            print(f'  Page {page} fetch error: {e}')
            break

        soup = BeautifulSoup(resp.text, 'lxml')

        # CSI's press release list items (adjust selectors if site updates)
        items = soup.select('div.press-release-item, article.news-item, li.press-release')
        if not items:
            # Fallback: look for links containing dates and titles
            items = soup.select('div[class*="press"], div[class*="news"], div[class*="release"]')

        if not items:
            print(f'  No items found on page {page}; stopping.')
            break

        found_on_page = 0
        for item in items:
            # Extract date
            date_tag = item.select_one('time, span[class*="date"], div[class*="date"]')
            raw_date = date_tag.get('datetime') or date_tag.get_text(strip=True) if date_tag else ''

            # Extract title and link
            link_tag = item.select_one('a[href]')
            if not link_tag:
                continue
            title = link_tag.get_text(strip=True)
            href  = link_tag['href']
            if not href.startswith('http'):
                href = BASE_URL + href

            results.append({'raw_date': raw_date, 'title': title, 'url': href})
            found_on_page += 1

        print(f'  Page {page}: {found_on_page} items')
        if found_on_page == 0:
            break

        page += 1
        time.sleep(0.8)  # polite crawl delay

    return results


session = requests.Session()
print('Fetching CSI press releases...')
all_releases = get_press_release_pages(session, PR_URL, max_pages=30)
print(f'Total press releases fetched: {len(all_releases)}')

## 3. Parse & Classify Acquisitions

In [ ]:
DATE_FORMATS = [
    '%Y-%m-%d', '%B %d, %Y', '%b %d, %Y',
    '%d %B %Y', '%d %b %Y', '%Y/%m/%d',
]

def parse_date(raw):
    raw = re.sub(r'(st|nd|rd|th),', ',', raw.strip())
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(raw, fmt).date()
        except ValueError:
            continue
    return None


def detect_operating_group(title):
    for group, aliases in OPERATING_GROUPS.items():
        for alias in aliases:
            if re.search(rf'\b{alias}\b', title, re.IGNORECASE):
                return group
    return 'Unknown'


def extract_target_name(title):
    """Best-effort extraction of acquired company name from press release title."""
    # e.g. "Harris acquires XYZ Software"
    patterns = [
        r'acqui(?:res?|sition of)\s+(.+?)(?:\s*[-–,]|$)',
        r'purchase(?:s|\s+of)?\s+(.+?)(?:\s*[-–,]|$)',
        r'completed?\s+acquisition\s+of\s+(.+?)(?:\s*[-–,]|$)',
    ]
    for pat in patterns:
        m = re.search(pat, title, re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return title  # fallback: full title


# Filter to acquisition-related press releases only
acquisitions = []
for r in all_releases:
    if not ACQ_PATTERN.search(r['title']):
        continue
    parsed_date = parse_date(r['raw_date'])
    acquisitions.append({
        'date':            parsed_date,
        'year':            parsed_date.year if parsed_date else None,
        'quarter':         f"Q{(parsed_date.month - 1) // 3 + 1} {parsed_date.year}" if parsed_date else None,
        'operating_group': detect_operating_group(r['title']),
        'target':          extract_target_name(r['title']),
        'title':           r['title'],
        'url':             r['url'],
    })

df = pd.DataFrame(acquisitions).dropna(subset=['date']).sort_values('date', ascending=False).reset_index(drop=True)
print(f'Acquisition press releases identified: {len(df)}')
df[['date', 'operating_group', 'target']].head(20)

## 4. Deal Cadence Analysis

In [ ]:
# Quarterly deal count
df['period'] = df['date'].apply(lambda d: pd.Period(d, freq='Q'))
quarterly = df.groupby('period').size().rename('deals').reset_index()
quarterly['period_str'] = quarterly['period'].astype(str)
quarterly = quarterly.sort_values('period')

# 4-quarter rolling average
quarterly['rolling_4q'] = quarterly['deals'].rolling(4, min_periods=1).mean()

print('Recent quarterly deal counts:')
print(quarterly.tail(12).to_string(index=False))

In [ ]:
# Annual deal count
annual = df.groupby('year').size().rename('deals').reset_index()
annual = annual[annual['year'] >= 2015].sort_values('year')

print('Annual deal counts (2015–present):')
print(annual.to_string(index=False))

In [ ]:
# Deals by operating group (last 12 months)
cutoff = date.today().replace(year=date.today().year - 1)
recent = df[df['date'] >= cutoff]

group_counts = recent.groupby('operating_group').size().rename('deals_ltm').sort_values(ascending=False)
print(f'Deals by operating group (LTM to {date.today()}):')
print(group_counts.to_frame().to_string())

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Constellation Software — Acquisition Deal Tracker', fontsize=14, fontweight='bold')

# 1. Quarterly deal cadence with rolling average
ax1 = axes[0, 0]
x = range(len(quarterly))
ax1.bar(x, quarterly['deals'], color='steelblue', alpha=0.7, label='Deals / quarter')
ax1.plot(x, quarterly['rolling_4q'], color='darkorange', linewidth=2, label='4Q rolling avg')
ax1.set_xticks(x[::4])
ax1.set_xticklabels(quarterly['period_str'].iloc[::4], rotation=45, ha='right', fontsize=8)
ax1.set_title('Quarterly Deal Cadence')
ax1.set_ylabel('Number of acquisitions')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. Annual deal count (bar)
ax2 = axes[0, 1]
ax2.bar(annual['year'].astype(str), annual['deals'], color='teal', alpha=0.75)
ax2.set_title('Annual Acquisitions (2015–Present)')
ax2.set_ylabel('Number of acquisitions')
ax2.set_xlabel('Year')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(axis='y', alpha=0.3)
for i, row in annual.iterrows():
    ax2.text(str(row['year']), row['deals'] + 0.5, str(int(row['deals'])), ha='center', fontsize=8)

# 3. Deals by operating group (LTM pie)
ax3 = axes[1, 0]
if len(group_counts) > 0:
    ax3.pie(group_counts.values, labels=group_counts.index, autopct='%1.0f%%',
            startangle=140, colors=plt.cm.Set3.colors)
    ax3.set_title('LTM Deals by Operating Group')
else:
    ax3.text(0.5, 0.5, 'No data', ha='center')
    ax3.set_title('LTM Deals by Operating Group')

# 4. Cumulative deals over time
ax4 = axes[1, 1]
df_sorted = df.sort_values('date')
df_sorted['cumulative'] = range(1, len(df_sorted) + 1)
ax4.plot(df_sorted['date'], df_sorted['cumulative'], color='navy', linewidth=1.5)
ax4.set_title('Cumulative Acquisitions Over Time')
ax4.set_ylabel('Cumulative count')
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax4.xaxis.set_major_locator(mdates.YearLocator(2))
ax4.tick_params(axis='x', rotation=45)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('csi_deal_tracker.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to csi_deal_tracker.png')

## 6. Investment Signal Summary

In [ ]:
# Compute current signal
last_4q_total = quarterly['deals'].tail(4).sum() if len(quarterly) >= 4 else quarterly['deals'].sum()
avg_per_quarter = last_4q_total / 4

ytd_year  = date.today().year
ytd_deals = df[df['year'] == ytd_year].shape[0]
ytd_day   = date.today().timetuple().tm_yday
ytd_run_rate = round(ytd_deals / ytd_day * 365)

# Signal classification
if avg_per_quarter > 25:
    signal = 'ACCELERATING'
    color  = '\033[92m'  # green
elif avg_per_quarter >= 15:
    signal = 'ON-PACE'
    color  = '\033[94m'  # blue
else:
    signal = 'SLOWING'
    color  = '\033[91m'  # red
reset = '\033[0m'

top_group = group_counts.idxmax() if len(group_counts) > 0 else 'N/A'
top_group_pct = round(group_counts.max() / group_counts.sum() * 100) if len(group_counts) > 0 else 0

print('=' * 55)
print(f'  CSI DEAL TRACKER — {date.today().strftime("%B %d, %Y")}')
print('=' * 55)
print(f'  Signal:              {color}{signal}{reset}')
print(f'  Avg deals / quarter: {avg_per_quarter:.1f}  (TTM {last_4q_total} deals)')
print(f'  {ytd_year} YTD deals:       {ytd_deals}  (annualized: ~{ytd_run_rate})')
print(f'  Most active group:   {top_group} ({top_group_pct}% of LTM deals)')
print(f'  Total tracked:       {len(df)} acquisitions')
print('=' * 55)

## 7. Recent Deals Table

In [ ]:
display_cols = ['date', 'operating_group', 'target', 'url']
recent_deals = df[display_cols].head(30)
recent_deals.columns = ['Date', 'Operating Group', 'Target / Title', 'Press Release']
recent_deals

## 8. Export to CSV

In [ ]:
out_path = 'csi_acquisitions.csv'
df.to_csv(out_path, index=False)
print(f'Exported {len(df)} records to {out_path}')